# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** OpenCode exited with status 0 without a valid queue-result.json. Last captured output: [2026-10-26T20:15:26.495Z] EOF | [2026-10-26T20:15:26.495Z] RUN=/home/phyto/phenotyping-colab/runs/p-d09aaac1e54e7ace6bc2213351a11bb3-20261026T194037Z; python3 $RUN/notes/build_notebook.py && python3 /home/phyto/phenotyping-colab/tools/validate_notebook.py $RUN/deliverable | [2026-10-26T20:15:26.495Z] Error: The user rejected permission to use this specific tool call. | Required queue-result.json is unavailable or invalid: FileNotFoundError

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: StomaAI : an efficient and user‐friendly tool for measurement of stomatal pores and density using deep computer vision

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-d09aaac1e54e7ace6bc2213351a11bb3)


# StomaAI: automated stomatal pore measurement with deep computer vision — minimal reproduction

**Paper:** Sai N., Bockman J.P., Chen H., Watson-Haigh N., Xu B., Feng X., Piechatzek A., Shen C., Gilliham M. (2023)
*StomaAI: an efficient and user-friendly tool for measurement of stomatal pores and density using deep computer vision.*
New Phytologist, doi:[10.1111/nph.18765](https://doi.org/10.1111/nph.18765)

**What this notebook does (scope):** it runs the authors' own **StomaAI v1.0** inference pipeline — unmodified, imported from the
pinned GitHub revision `92a21c9d283fed6aed253188e3590de5c4b16990` — on **one deposited example image** (Arabidopsis, *"1 - At"*, from the authors' Figshare deposit),
using the paper-version Detectron2 weights and configs. It then compares the measurements with (a) the authors' deposited SAI
predictions for the same image and (b) the deposited human ground-truth annotations (`gt.json`, SAI Assets v2.0).

**What this is *not*:** this single-example run does not verify the paper's dataset-level results (35-image human/machine test,
Xu et al. replication, mAP evaluation). Training is out of scope (no public training annotations). The Streamlit GUI is not
reproduced; the same scientific inference/measure/draw modules are used headlessly.

**Author code reuse:** all model/inference/measurement code is the authors' GPL-3.0 code from
[XDynames/SAI-app](https://github.com/XDynames/SAI-app) (v1.0), imported from a pinned checkout inside this Colab VM —
not re-implemented. Glue code (downloads, checksums, tables, plots) is notebook-specific.

**日本語:** このノートブックは、著者自身の StomaAI v1.0 推論コードをピン留めされたリビジョンからそのまま呼び出し、
著者が公開している Arabidopsis の例画像 1 枚（"1 - At"）に実行します。出力した気孔計測値を、著者のデポジットした
SAI 予測 JSON と人手アノテーション（gt.json）と比較します。単一画像のデモであり、論文のデータセット全体の精度
検証ではありません。学習は対象外です。

**Recommended runtime:** `Runtime > Change runtime type > T4 GPU` (fastest). A **CPU runtime also works** (the authors' code
auto-detects the device; the paper reports 6–12 s/image on a mid-range CPU). Estimated total: **~10–20 min** (mostly the
one-time Detectron2 build) + **~390 MB** weight download.


## 1. Environment check

Reports the allocated Colab hardware. StomaAI's own code selects CUDA or CPU automatically
(`inference/infer.py: setup_model_configuration`), so both runtime types run the same scientific path.

**日本語:** まず実行環境（GPU の有無）を確認します。著者コードが自動で CPU/CUDA を切り替えるため、どちらのランタイムでも同じ処理が動きます。

In [ ]:
import os, sys, platform
print("Python:", sys.version.split()[0])
import torch, torchvision
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU runtime: inference will run on CPU (faithful to the authors' device auto-detection)")

## 2. Dependencies

Installs Detectron2 (built from the official source against the preinstalled torch) plus the packages used by the author
modules on this path: `streamlit` (imported at module level by the authors' app code), `shapely`, `rasterio`
(`rasterio.transform.IDENTITY` is used by their mask→polygon step), `mask-to-polygons==0.0.2 --no-deps` (exactly as in the
authors' `setup.sh`), and `pandas/scipy/pyyaml`. `opencv-python` is preinstalled on Colab. Not installed (not on the executed
path): `geojson`, `watchdog`. This cell takes about 5–15 minutes (Detectron2 CUDA build).

**日本語:** Detectron2 を公式ソースからビルドし、著者モジュールが必要とする依存関係をインストールします（約 5〜15 分）。

In [ ]:
%pip install --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git"
%pip install streamlit shapely rasterio pandas scipy pyyaml
%pip install --no-deps "mask-to-polygons==0.0.2"

Verify that every import used by the author code path resolves on this runtime before touching the model.

**日本語:** 著者コードが使う全モジュールが import できるかを確認します。

In [ ]:
import detectron2, cv2, shapely, rasterio, yaml, pandas, scipy, matplotlib, streamlit
import mask_to_polygons
from mask_to_polygons.vectorification import geometries_from_mask  # author app/inference.py import
from rasterio.transform import IDENTITY                            # author app/inference.py import
print("detectron2:", detectron2.__version__)
print("cv2:", cv2.__version__, "| shapely:", shapely.__version__, "| pandas:", pandas.__version__)
print("mask_to_polygons:", getattr(mask_to_polygons, "__version__", "0.0.2"), "| streamlit:", streamlit.__version__)
print("All author-path dependencies importable.")

## 3. Get the authors' pinned code (v1.0)

Partial, blob-less clone of `XDynames/SAI-app` **inside this Colab VM**, fetching exactly the v1.0 commit `92a21c9d283fed6aed253188e3590de5c4b16990` in detached
HEAD (the release the authors tie to this paper). The checkout is asserted against the pinned SHA; only the `inference`, `app`,
`tools`, `interface` package directories are materialised. All repository bytes stay under `/content`.

**日本語:** 著者リポジトリをピン留めしたコミットで部分クローンし、コミットが一致することを検証してから import パスに追加します。

In [ ]:
import subprocess, pathlib
REPO_URL = "https://github.com/XDynames/SAI-app.git"
PIN = "92a21c9d283fed6aed253188e3590de5c4b16990"
REPO_DIR = "/content/SAI-app"
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, REPO_DIR], check=True)
def git(*args):
    return subprocess.run(["git", "-C", REPO_DIR, *args], check=True, capture_output=True, text=True).stdout.strip()
git("sparse-checkout", "init", "--cone")
git("sparse-checkout", "set", "inference", "app", "tools", "interface")
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", PIN], check=True)
git("checkout", "--detach", "-f", PIN)
head = git("rev-parse", "HEAD")
assert head == PIN, f"checkout mismatch: {head} != {PIN}"
print("SAI-app checked out at", head)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 4. Register the custom SAI model heads

StomaAI's config selects custom Detectron2 registry names (`KPROIHeads`, `KRCNNConvHead`) implementing the paper's
raised-resolution keypoint/segmentation heads on a non-square pooler (`inference/modeling/stoma_head.py`, `poolers.py`).
The authors' `main.py` imports this module before building the model — we do exactly the same, then assert registration.

**日本語:** 論文独自の高解像度キーポイント/セグメンテーションヘッド（著者の `stoma_head.py`）を Detectron2 レジストリに登録します（`main.py` と同じ手順）。

In [ ]:
from inference.modeling.stoma_head import KRCNNConvHead, KPROIHeads  # noqa: F401  (registers heads, as main.py does)
from detectron2.modeling import ROI_HEADS_REGISTRY, ROI_KEYPOINT_HEAD_REGISTRY
assert "KPROIHeads" in ROI_HEADS_REGISTRY, "KPROIHeads not registered"
assert "KRCNNConvHead" in ROI_KEYPOINT_HEAD_REGISTRY, "KRCNNConvHead not registered"
print("Custom SAI heads registered: KPROIHeads, KRCNNConvHead")

## 5. Download paper assets (checksummed) — inside Colab only

Downloads into `/content/assets` using the **authors' own pinned Figshare IDs** (`tools/cloud_files.py`):
`Arabidopsis.yaml` + `Base-RCNN-FPN.yaml` (v1.0 configs) and `arabidopsis.pth` (paper-version weights, 389 MB) from the
*"SAI Assets"* deposit (doi:10.25909/25115054.v6), plus the example image *"1 - At"*, the authors' deposited SAI predictions,
and the human ground truth `gt.json` (*"SAI Assets v2.0"*, doi:10.25909/25894681.v32). Every file is verified against the
size and MD5 published in the Figshare file metadata; a mismatch aborts.

**日本語:** 論文の重み・設定・例画像・著者の予測 JSON・正解 JSON を著者がピン留めした Figshare ID からダウンロードし、
サイズと MD5 を検証します（不一致なら中断します）。

In [ ]:
import hashlib, pathlib, requests
from tools.cloud_files import EXTERNAL_DEPENDANCIES
from tools.load import download_and_save_yaml, download_and_save_model_weights

ASSETS = pathlib.Path("/content/assets")
(ASSETS / "config").mkdir(parents=True, exist_ok=True)
(ASSETS / "arabidopsis").mkdir(parents=True, exist_ok=True)

# Authors' own helpers (tools/load.py) for the two YAMLs and the weights
download_and_save_yaml(EXTERNAL_DEPENDANCIES["Arabidopsis_config"], "./assets/config/Arabidopsis.yaml")
download_and_save_yaml(EXTERNAL_DEPENDANCIES["base_config"], "./assets/config/Base-RCNN-FPN.yaml")
download_and_save_model_weights(EXTERNAL_DEPENDANCIES["Arabidopsis_weights"], "./assets/arabidopsis/weights.pth")

def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

CHECKSUMS = {  # name: (relpath, expected_bytes, expected_md5) -- from Figshare file metadata
    "Arabidopsis.yaml":   ("assets/config/Arabidopsis.yaml",   696,        "45f98a146228021ee0a70fb0c39ea1ce"),
    "Base-RCNN-FPN.yaml": ("assets/config/Base-RCNN-FPN.yaml", 1130,       "16a2514297396b47ffb8c72d1321991d"),
    "arabidopsis.pth":    ("assets/arabidopsis/weights.pth",   388873763,  "4fccd96c460c542765fa5743570d778b"),
}
for name, (rel, size, md5) in CHECKSUMS.items():
    p = pathlib.Path(rel)
    assert p.stat().st_size == size, f"{name}: size {p.stat().st_size} != {size}"
    assert file_md5(p) == md5, f"{name}: md5 mismatch"
    print(f"OK  {name}  {size} bytes  md5={md5}")

# Data files (example image + deposited prediction/gt JSONs), fetched here in Colab only
DATA_FILES = {
    "1 - At.jpg":     (f"https://adelaide.figshare.com/ndownloader/files/44323415", 687591, "d3854df91032d3403d30a1b4a3ba86a8", "assets/arabidopsis/1 - At.jpg"),
    "1 - At.json":    (f"https://adelaide.figshare.com/ndownloader/files/44323715", 11657,  "3e09489458242dee6510664a5c82fd6e", "assets/arabidopsis/1 - At.json"),
    "1 - At gt.json": (f"https://adelaide.figshare.com/ndownloader/files/46635838", 8678,   "76f062c778d3eb87d28b90b6b0d0100b", "assets/arabidopsis/1 - At gt.json"),
}
for name, (url, size, md5, rel) in DATA_FILES.items():
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    assert len(r.content) == size, f"{name}: size {len(r.content)} != {size}"
    assert hashlib.md5(r.content).hexdigest() == md5, f"{name}: md5 mismatch"
    (ASSETS.parent / rel).write_bytes(r.content)
    print(f"OK  {name}  {size} bytes  md5={md5}")

## 6. Input preview

Loads the example the way the authors' pipeline does (OpenCV BGR via `cv2.imread`) and shows the raw microscopy image:
Arabidopsis *"1 - At"*, downloaded above from the authors' Figshare deposit.

**日本語:** 例画像を読み込み、推論の前に表示します。

In [ ]:
import cv2, matplotlib.pyplot as plt
IMAGE_PATH = "/content/assets/arabidopsis/1 - At.jpg"
image_bgr = cv2.imread(IMAGE_PATH)  # authors' pipeline uses OpenCV BGR images
assert image_bgr is not None, f"failed to read {{IMAGE_PATH}}"
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
IMG_H, IMG_W = image_rgb.shape[:2]
print("Input image:", IMAGE_PATH, "| shape (H, W, C):", image_rgb.shape)
fig, ax = plt.subplots(figsize=(6, 5))
ax.axis("off")
ax.imshow(image_rgb)
ax.set_title(f'Input: Arabidopsis "1 - At" ({IMG_W}x{IMG_H} px)\nSAI Assets, doi:10.25909/25115054.v6')
plt.show()

## 7. Build the inference engine (authors' code path)

`setup_inference_engine("Arabidopsis")` is the authors' own function (`inference/infer.py`): it skips the downloads we already
verified, merges their Figshare-hosted YAML configs, applies their defaults — score threshold **0.6** (their v1.0
`BASE_CONFIDENCE_THRESHOLD`) and the keypoint-pooler fix `POOLER_RESOLUTION=(14,14)` — and selects CUDA or CPU automatically.

**日本語:** 著者自身の関数で推論エンジンを構築します（信頼度しきい値 0.6、デバイスは自動選択）。

In [ ]:
os.chdir("/content")  # author helpers write ./assets relative to the working directory
from tools.state import Option_State
from inference.infer import setup_inference_engine, Inference_Engines

Option_State["plant_type"] = "Arabidopsis"
setup_inference_engine("Arabidopsis")
engine = Inference_Engines["Arabidopsis"]
print("Model device:", engine.predictor.cfg.MODEL.DEVICE)
print("Score threshold:", engine.predictor.cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST)
print("Keypoint pooler resolution:", engine.predictor.cfg.MODEL.ROI_KEYPOINT_HEAD.POOLER_RESOLUTION)

## 8. Run StomaAI inference on the example

`run_on_image` is the authors' entry point: a Detectron2 `DefaultPredictor` forward pass, then their validity post-processing
(IoU-based duplicate removal, near-edge and size filters — `inference/post_processing.py`). Class 1 = open pore (polygon +
keypoints), class 0 = closed pore.

**日本語:** 著者のエントリポイントで推論を実行し、フィルタ後の有効検出数を報告します。

In [ ]:
from inference.infer import run_on_image
instances, elapsed, valid_indices = run_on_image(image_bgr)
n_raw = len(instances.pred_boxes)
n_valid = len(valid_indices)
print(f"Inference time: {elapsed:.2f} s")
print(f"Raw detections: {n_raw} | valid after SAI post-processing: {n_valid}")
if n_raw == 0:
    raise RuntimeError("No detections at all, refusing to continue with an empty result")


## 9. Extract pore measurements (authors' measurement code)

`predictions_to_list_of_dictionaries` (the authors' `app/inference.py`) converts each instance into a record: pore **length**
from the predicted endpoint keypoints A–B, pore **width** from the polygon intersection C–D perpendicular to A–B, pore **area**
as the predicted mask pixel count, plus open/closed class and confidence — all in pixels. We convert to µm with the authors'
calibration constant (`CAMERA_CALIBRATION["Arabidopsis"] = 10.25131 px/µm`, `tools/constants.py`), applying the same double
division for area that the authors' `convert_measurements` uses (px² → µm²).

**日本語:** 著者の計測コードで気孔の長さ・幅・面積（ピクセル）を抽出し、著者の較準定数 10.25131 px/µm で µm 単位に変換します。

In [ ]:
import pandas as pd
from app.inference import predictions_to_list_of_dictionaries
from tools.constants import CAMERA_CALIBRATION
from interface.upload_single import convert_to_SIU_length

CAL = CAMERA_CALIBRATION["Arabidopsis"]  # px per micron (authors' tools/constants.py)
Option_State["camera_calibration"] = CAL
detections, valid_idx = predictions_to_list_of_dictionaries(instances, valid_indices)

rows = []
for d in detections:
    rows.append({
        "stoma_id": d["stoma_id"],
        "valid": d["stoma_id"] in valid_idx,
        "class": "open" if d["category_id"] else "closed",
        "length_um": convert_to_SIU_length(d["length"]),
        "width_um": convert_to_SIU_length(d["width"]) if d["category_id"] == 1 else float("nan"),
        "area_um2": convert_to_SIU_length(convert_to_SIU_length(d["area"])),  # authors apply conversion twice for area
        "width_over_length": d["width"] / d["length"] if d["length"] > 0 else 0.0,
        "confidence": d["confidence"],
    })
df = pd.DataFrame(rows).sort_values("stoma_id").reset_index(drop=True)
valid_df = df[df["valid"]].reset_index(drop=True)
assert len(valid_df) > 0, "No valid detections after SAI post-processing."
print(f"{len(valid_df)} valid stomata (of {len(df)} raw detections). Measurements in micrometres:")
display(valid_df.round(3))

## 10. Summary statistics, stomatal density, CSV export

Follows the authors' summary module (`app/summary_statistics.py`): pore count, mean length/width/area in µm, and density as
stomata per mm² using the image dimensions and the calibration. The authors' `IMAGE_AREA` constant for this camera is shown
as a cross-check. The measurement table is also saved as CSV inside the Colab VM (`/content/sai_pore_measurements.csv`,
downloadable from the file browser).

**日本語:** 有効気孔数・平均長/幅/面積・気孔密度（個/mm²）を整理し、CSV に保存します。

In [ ]:
from tools.constants import IMAGE_AREA
img_area_mm2 = (IMG_H / CAL / 1000) * (IMG_W / CAL / 1000)
summary = pd.DataFrame([
    {"property": "Stomata count (valid)", "this run": len(valid_df)},
    {"property": "Mean pore length (µm)", "this run": round(valid_df["length_um"].mean(), 2)},
    {"property": "Mean pore width (µm)", "this run": round(valid_df["width_um"].mean(), 2)},
    {"property": "Mean pore area (µm²)", "this run": round(valid_df["area_um2"].mean(), 2)},
    {"property": "Image area (mm²)", "this run": round(img_area_mm2, 6)},
    {"property": "Stomatal density (stomata/mm²)", "this run": round(len(valid_df) / img_area_mm2, 2)},
    {"property": "Authors' IMAGE_AREA constant (mm²)", "this run": IMAGE_AREA["Arabidopsis"]},
])
display(summary)
valid_df.to_csv("/content/sai_pore_measurements.csv", index=False)
print("Saved: /content/sai_pore_measurements.csv")

## 11. Compare with the authors' deposited SAI predictions

The authors deposited SAI's own predictions for this exact image (*"1 - At.json"*, SAI Assets). First the schema is inspected
(read from the file itself, not assumed), including the bounding-box convention (corners `x1,y1,x2,y2` vs `x,y,w,h`).

**日本語:** 著者がデポジットした同一画像の SAI 予測 JSON を読み込み、スキーマと bbox 形式を確認します。

In [ ]:
import json
with open("/content/assets/arabidopsis/1 - At.json") as f:
    author_preds = json.load(f)
print("Top-level keys:", sorted(author_preds.keys()))
dep = author_preds.get("detections", [])
print("Deposited detections:", len(dep))
print("First-detection keys:", sorted(dep[0].keys()) if dep else "n/a")
dep_valid = author_preds.get("valid_detection_indices", None)
print("valid_detection_indices present:", dep_valid is not None, "" if dep_valid is None else f"(n={len(dep_valid)})")
if dep:
    b = dep[0]["bbox"]
    DEP_BBOX_IS_CORNERS = b[2] > b[0] and b[3] > b[1]
    print("Deposited bbox appears to be x1,y1,x2,y2 corners:", DEP_BBOX_IS_CORNERS)

Deposited records, like ours, are pixel-unit SAI outputs. Every **valid** detection of our run is matched to the deposited
detection with the highest bounding-box IoU (kept only if IoU > 0.5, using the authors' own `intersection_over_union`), and the
pore lengths are compared. Unmatched entries are reported honestly.

**日本語:** 自分の実行結果の有効検出を IoU>0.5 で著者の検出と対応付け、気孔長の差（px と µm）を集計します。

In [ ]:
import numpy as np
from inference.post_processing import intersection_over_union

def dep_bbox_corners(rec):
    b = list(rec["bbox"])
    if not (b[2] > b[0] and b[3] > b[1]):  # stored as x,y,w,h
        b = [b[0], b[1], b[0] + b[2], b[1] + b[3]]
    return b

matched, ious, diffs_px = 0, [], []
for _, row in valid_df.iterrows():
    ours = detections[int(row["stoma_id"])]["bbox"]
    best_iou, best_det = 0.0, None
    for d in dep:
        iou = intersection_over_union(ours, dep_bbox_corners(d))
        if iou > best_iou:
            best_iou, best_det = iou, d
    if best_det is not None and best_iou > 0.5:
        matched += 1
        ious.append(best_iou)
        diffs_px.append(abs(row["length_um"] * CAL - best_det["length"]))
print(f"Deposited detections: {len(dep)} | our valid: {len(valid_df)}")
if ious:
    diffs_um = [d / CAL for d in diffs_px]
    print(f"IoU-matched: {matched} / {len(valid_df)} valid (mean IoU {np.mean(ious):.3f})")
    print(f"Pore length |diff| vs deposited: mean {np.mean(diffs_px):.3f} px ({np.mean(diffs_um):.3f} µm), max {np.max(diffs_px):.3f} px ({np.max(diffs_um):.3f} µm)")
else:
    print("No IoU-matched detections.")

## 12. Compare with the deposited human ground truth

`gt.json` comes from the authors' *"SAI Assets v2.0"* deposit (the replacement for the deprecated CloudStor links, per the
SAI-app v2.0 release notes). The schema is inspected from the file itself, then the human stomatal count/density are put next
to our run's — a count/agreement check, not the paper's full CCC analysis (which used 4 experts over 35 images).

**日本語:** 人手アノテーション gt.json のスキーマを確認し、気孔数と密度を本実行結果と比較します（単一画像の一致確認であり、
論文の CCC 解析の再現ではありません）。

In [ ]:
with open("/content/assets/arabidopsis/1 - At gt.json") as f:
    gt = json.load(f)
print("Top-level keys:", sorted(gt.keys()))
gdet = gt.get("detections", [])
print("Ground-truth detections:", len(gdet))
print("First-detection keys:", sorted(gdet[0].keys()) if gdet else "n/a")

The summary below lists, side by side: our run, the authors' deposited SAI predictions, and the human annotations — count,
mean pore length (when the record contains it), and density per mm². Missing fields are reported as `n/a` rather than invented.

**日本語:** 本実行・著者の SAI 予測・人手アノテーションの要約（件数・平均気孔長・密度）を並べて表示します。

In [ ]:
def mean_length_px(records):
    vals = [r["length"] for r in records if "length" in r and isinstance(r["length"], (int, float)) and r["length"] > 0]
    return float(np.mean(vals)) if vals else None

our_density = len(valid_df) / img_area_mm2
dep_density = len(dep) / img_area_mm2
gt_density = len(gdet) / img_area_mm2
dep_mean, gt_mean = mean_length_px(dep), mean_length_px(gdet)
comparison = pd.DataFrame([
    {"source": "This run (SAI v1.0, valid detections)", "stomata": len(valid_df),
     "mean length (µm)": round(valid_df["length_um"].mean(), 2), "density (/mm²)": round(our_density, 1)},
    {"source": "Authors' deposited SAI predictions", "stomata": len(dep),
     "mean length (µm)": round(dep_mean / CAL, 2) if dep_mean else "n/a", "density (/mm²)": round(dep_density, 1)},
    {"source": "Human ground truth (gt.json, v2.0 deposit)", "stomata": len(gdet),
     "mean length (µm)": round(gt_mean / CAL, 2) if gt_mean else "n/a", "density (/mm²)": round(gt_density, 1)},
])
display(comparison)

## 13. Overlay visualization: predictions vs ground truth

Left: the original input with **this run's** valid SAI predictions (blue masks/length keypoints, green boxes = open pores,
orange boxes = closed pores). Right: the same input with the **deposited human annotations** (red, SAI Assets v2.0
`gt.json`). Both panels use the authors' own drawing module (`tools/draw.py`); the gt panel adapts to whichever annotation
fields the deposited file actually contains.

**日本語:** 左に本実行の予測（青: マスク/キーポイント, 緑: 開口, 橙: 閉鎖）、右に人手アノテーション（赤）を重ねた図を作成します。

In [ ]:
from tools import draw
valid_dets = [detections[int(i)] for i in valid_idx]

def flat_seg(rec):
    seg = rec.get("segmentation")
    if seg and isinstance(seg[0], list):  # COCO-style nested polygon -> authors' flat format
        rec = dict(rec)
        rec["segmentation"] = [ [float(x) for x in seg[0]] ]
    return rec

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].axis("off"); axes[0].imshow(image_rgb)
draw.masks(axes[0], valid_dets, False)
draw.keypoints(axes[0], valid_dets, False)
draw.bboxes(axes[0], valid_dets, False)
axes[0].set_title("SAI predictions (this run)\nblue = masks & length keypoints, green = open, orange = closed", fontsize=9)
axes[1].axis("off"); axes[1].imshow(image_rgb)
has_segmentation = bool(gdet) and "segmentation" in gdet[0] and gdet[0]["segmentation"]
has_keypoints = bool(gdet) and "keypoints" in gdet[0]
if has_segmentation:
    draw.masks(axes[1], [flat_seg(r) for r in gdet], True)
if has_keypoints:
    draw.keypoints(axes[1], gdet, True)
if (not has_segmentation) and (not has_keypoints) and bool(gdet):
    draw.bboxes(axes[1], gdet, True)
axes[1].set_title(f"Human annotations (gt.json)\nfields used: {sorted(gdet[0].keys()) if gdet else 'none'}", fontsize=9)
plt.tight_layout()
plt.show()

## 14. Interpretation, scope, and validation record

**How to read the results above.** The inference-time cell reports the measured speed for the allocated runtime (a T4 GPU
should be fast; a CPU runtime takes tens of seconds — the paper reports 6–12 s/image on a mid-range CPU). The measurement
table lists every valid stoma with class and µm measurements; the comparison tables show how closely this run reproduces the
authors' deposited predictions, and how predictions relate to the human count/mean length/density on this single image.
Closed pores (class 0) have no width/area by design — the authors define width only for open pores with a polygon.

**Scope and limitations.** One image, one species model (Arabidopsis). This is a faithful execution of the authors' own v1.0
inference path, not a reproduction of the paper's statistical results (CCC vs 4 experts on 35 images, mAP, Xu et al.
replication). Score threshold 0.6 follows the authors' v1.0 code (`BASE_CONFIDENCE_THRESHOLD`); the published abstract mentions
0.5 for the web demo. All data was downloaded inside this VM from the authors' checksummed Figshare deposits.

**Provenance.** Code: XDynames/SAI-app @ `92a21c9d283fed6aed253188e3590de5c4b16990` (v1.0, GPL-3.0). Weights/configs/example:
doi:10.25909/25115054.v6 (arabidopsis.pth 44321711, Arabidopsis.yaml 44321570, Base-RCNN-FPN.yaml 44321576, "1 - At.jpg"
44323415, "1 - At.json" 44323715). Ground truth: doi:10.25909/25894681.v32 ("1 - At gt.json", 46635838). Calibration and
area constants: `tools/constants.py` (10.25131 px/µm; 0.04794822 mm²), consistent with the 2592×1944 input size.

**日本語のまとめ:** 本ノートブックは著者の v1.0 コードをそのまま実行し、Arabidopsis 例画像 1 枚の気孔を検出・計測して
µm 単位の表と密度を出力し、著者のデポジット予測および人手アノテーションと比較します。単一画像のデモであり、論文の
統計的検証（CCC・mAP・Xu et al. の再現）は対象外です。閉鎖気孔には幅・面積は定義されません。

**References.**
1. Sai N. et al. (2023) New Phytologist, doi:10.1111/nph.18765.
2. SAI-app code, GPL-3.0: https://github.com/XDynames/SAI-app (v1.0).
3. SAI Assets (v1): doi:10.25909/25115054.v6 · SAI Assets v2.0: doi:10.25909/25894681.v32.
